# Feature Engineering Training

Jalankan setelah Combination selesai. GridSearchCV membandingkan tiga embedding dan tujuh kelompok fitur pada validation tetap; model akhir tujuh fitur selalu disimpan. Ini hanya memakai training resmi.

## 1. Konfigurasi
Siapkan folder model dan laporan untuk run gabungan.

In [1]:
from pathlib import Path
import hashlib
import json
import time

import joblib
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, PredefinedSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.svm import SVC

RUN_ID = "cifar10_7fitur_v1"
SEED = 30092026
ROOT = Path.cwd().resolve()
OUT = ROOT / "outputs" / RUN_ID / "fusion"

for folder in ("model", "fitur", "laporan"):
    (OUT / folder).mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

I0000 00:00:1791489388.509445  208307 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791489388.549391  208307 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791489389.511365  208307 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 2. Data training
Muat 50.000 label training resmi dan hitung split tetap.

In [2]:
from tensorflow.keras.datasets import cifar10

(x_all, y_all), _ = cifar10.load_data()
y_all = y_all.ravel().astype(np.int64)
assert x_all.shape == (50000, 32, 32, 3)

train_idx, val_idx = train_test_split(
    np.arange(len(y_all)),
    test_size=0.2,
    stratify=y_all,
    random_state=SEED,
)
train_idx = np.sort(train_idx)
val_idx = np.sort(val_idx)
assert len(train_idx) == 40000
assert len(val_idx) == 10000

split_hash = hashlib.sha256(
    train_idx.tobytes() + val_idx.tobytes()
).hexdigest()
print("Split:", len(train_idx), len(val_idx), split_hash[:16])

Split: 40000 10000 16004a1ac8a89ac9


## 3. Muat fitur gabungan
Validasi dimensi tujuh blok, urutan, label, indeks, split, dan hash CNN sumber.

In [3]:
with np.load(OUT / "fitur" / "train_combined.npz", allow_pickle=False) as archive:
    assert str(archive["run_id"]) == RUN_ID and str(archive["split_hash"]) == split_hash
    assert str(archive["preprocessing_version"]) == "v1"
    assert np.array_equal(archive["indices"], np.arange(50000))
    assert np.array_equal(archive["y"], y_all)
    assert np.array_equal(archive["train_idx"], train_idx)
    assert np.array_equal(archive["val_idx"], val_idx)
    assert tuple(archive["block_names"].tolist()) == (
        "rgb", "avg", "ntsc", "hog", "lbp", "hsv", "rgb_stats"
    )
    boundaries = archive["boundaries"].astype(np.int32)
    assert np.array_equal(boundaries, [0, 512, 1024, 1536, 1860, 2116, 2164, 2173])
    model_hashes = archive["model_hashes"].tolist()
    combined = archive["features"].astype(np.float32)
assert combined.shape == (50000, 2173) and np.isfinite(combined).all()
for variant, expected_hash in zip(("rgb", "avg", "ntsc"), model_hashes):
    model_path = ROOT / "outputs" / RUN_ID / variant / "model" / "cnn.keras"
    assert hashlib.sha256(model_path.read_bytes()).hexdigest() == expected_hash

## 4. Pembobot blok
Setelah StandardScaler, empat descriptor tambahan dikalikan bobot terpilih; tiga embedding tetap berbobot 1.

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
class BlockWeight(TransformerMixin, BaseEstimator):
    def __init__(self, extra_weight=1.0, embedding_dim=1536):
        self.extra_weight, self.embedding_dim = extra_weight, embedding_dim
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        result = np.asarray(X, dtype=np.float32).copy()
        result[:, self.embedding_dim:] *= self.extra_weight
        return result

## 5. Grid SVM tiga embedding
Pembanding 1536 dimensi memakai empat kombinasi C dan gamma. Scaler di-fit hanya pada training fold.

In [5]:
fold = np.full(len(y_all), -1, dtype=np.int8)
fold[val_idx] = 0

baseline_pipeline = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("norm", Normalizer()),
        ("svc", SVC(kernel="rbf", cache_size=512, random_state=SEED, verbose=1)),
    ]
)
baseline_grid = GridSearchCV(
    estimator=baseline_pipeline,
    param_grid={
        "svc__C": [1, 10],
        "svc__gamma": ["scale", 0.001],
    },
    cv=PredefinedSplit(fold),
    refit=False,
    n_jobs=1,
    error_score="raise",
    verbose=1,
)

baseline_start = time.perf_counter()
baseline_grid.fit(combined[:, :1536], y_all)
baseline_seconds = time.perf_counter() - baseline_start
print("Akurasi validation tiga embedding:", baseline_grid.best_score_)
print("Parameter terbaik:", baseline_grid.best_params_)

Fitting 1 folds for each of 4 candidates, totalling 4 fits
[LibSVM]*
optimization finished, #iter = 117
obj = -5.110520, rho = 0.005301
nSV = 50, nBSV = 0
*
optimization finished, #iter = 99
obj = -8.478024, rho = -0.006621
nSV = 51, nBSV = 5
*
optimization finished, #iter = 113
obj = -6.391495, rho = 0.126591
nSV = 50, nBSV = 1
*
optimization finished, #iter = 95
obj = -5.772299, rho = 0.068672
nSV = 42, nBSV = 0
*
optimization finished, #iter = 104
obj = -4.387479, rho = -0.121870
nSV = 51, nBSV = 0
*
optimization finished, #iter = 121
obj = -5.037144, rho = 0.000695
nSV = 50, nBSV = 0
*
optimization finished, #iter = 118
obj = -6.513807, rho = 0.122268
nSV = 59, nBSV = 1
*
optimization finished, #iter = 118
obj = -5.810233, rho = 0.023031
nSV = 57, nBSV = 0
*
optimization finished, #iter = 119
obj = -5.491531, rho = -0.072014
nSV = 53, nBSV = 0
*
optimization finished, #iter = 100
obj = -5.645133, rho = 0.026003
nSV = 53, nBSV = 2
*
optimization finished, #iter = 125
obj = -7.574570

## 6. Grid SVM tujuh fitur
Uji bobot fitur tambahan 0.25 dan 1.0 bersama C dan gamma. Validation tidak ikut fitting.

In [6]:
fusion_pipeline = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("weight", BlockWeight()),
        ("norm", Normalizer()),
        ("svc", SVC(kernel="rbf", cache_size=512, random_state=SEED, verbose=1)),
    ]
)
fusion_grid = GridSearchCV(
    estimator=fusion_pipeline,
    param_grid={
        "weight__extra_weight": [0.25, 1.0],
        "svc__C": [1, 10],
        "svc__gamma": ["scale", 0.001],
    },
    cv=PredefinedSplit(fold),
    refit=False,
    n_jobs=1,
    error_score="raise",
    verbose=1,
)

fusion_start = time.perf_counter()
fusion_grid.fit(combined, y_all)
fusion_seconds = time.perf_counter() - fusion_start
print("Akurasi validation tujuh fitur:", fusion_grid.best_score_)
print("Parameter terbaik:", fusion_grid.best_params_)

Fitting 1 folds for each of 8 candidates, totalling 8 fits
[LibSVM]*
optimization finished, #iter = 138
obj = -6.310931, rho = 0.029430
nSV = 75, nBSV = 0
*
optimization finished, #iter = 141
obj = -9.795859, rho = 0.043702
nSV = 74, nBSV = 4
*
optimization finished, #iter = 148
obj = -7.589452, rho = 0.131147
nSV = 77, nBSV = 0
*
optimization finished, #iter = 114
obj = -6.717453, rho = 0.050970
nSV = 64, nBSV = 0
*
optimization finished, #iter = 143
obj = -5.428428, rho = -0.101789
nSV = 80, nBSV = 0
*
optimization finished, #iter = 129
obj = -6.140491, rho = 0.013516
nSV = 73, nBSV = 0
*
optimization finished, #iter = 149
obj = -7.437875, rho = 0.121372
nSV = 82, nBSV = 1
*
optimization finished, #iter = 156
obj = -6.982736, rho = 0.011677
nSV = 86, nBSV = 0
*
optimization finished, #iter = 147
obj = -6.794020, rho = -0.034038
nSV = 77, nBSV = 0
*
optimization finished, #iter = 134
obj = -7.010527, rho = 0.023598
nSV = 82, nBSV = 2
*
optimization finished, #iter = 140
obj = -9.13216

## 7. Fit SVM pembanding
Gunakan hanya 40.000 training dan parameter terpilih.

In [7]:
baseline_svm = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("norm", Normalizer()),
        (
            "svc",
            SVC(
                kernel="rbf",
                C=baseline_grid.best_params_["svc__C"],
                gamma=baseline_grid.best_params_["svc__gamma"],
                cache_size=512,
                random_state=SEED,
                verbose=1,
            ),
        ),
    ]
)
baseline_svm.fit(combined[train_idx, :1536], y_all[train_idx])
baseline_val = float(
    accuracy_score(
        y_all[val_idx],
        baseline_svm.predict(combined[val_idx, :1536]),
    )
)
print("Validation tiga embedding:", baseline_val)

[LibSVM]*
optimization finished, #iter = 700
obj = -753.428544, rho = -0.053586
nSV = 1396, nBSV = 1394
*
optimization finished, #iter = 836
obj = -928.818104, rho = -0.063944
nSV = 1670, nBSV = 1668
*
optimization finished, #iter = 733
obj = -805.482798, rho = -0.029203
nSV = 1466, nBSV = 1464
*
optimization finished, #iter = 702
obj = -765.498871, rho = -0.061674
nSV = 1404, nBSV = 1402
*
optimization finished, #iter = 697
obj = -734.705794, rho = -0.042247
nSV = 1369, nBSV = 1365
*
optimization finished, #iter = 672
obj = -719.826617, rho = -0.052048
nSV = 1337, nBSV = 1333
*
optimization finished, #iter = 718
obj = -784.647584, rho = -0.037051
nSV = 1436, nBSV = 1434
*
optimization finished, #iter = 843
obj = -929.360107, rho = -0.051804
nSV = 1686, nBSV = 1684
*
optimization finished, #iter = 767
obj = -836.299834, rho = -0.038101
nSV = 1532, nBSV = 1530
*
optimization finished, #iter = 663
obj = -710.302779, rho = 0.018185
nSV = 1320, nBSV = 1318
*
optimization finished, #iter = 

## 8. Fit SVM tujuh fitur
Model akhir selalu memakai ketujuh kelompok dan tetap dilatih hanya pada 40.000 training.

In [8]:
fusion_svm = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        (
            "weight",
            BlockWeight(
                extra_weight=fusion_grid.best_params_["weight__extra_weight"]
            ),
        ),
        ("norm", Normalizer()),
        (
            "svc",
            SVC(
                kernel="rbf",
                C=fusion_grid.best_params_["svc__C"],
                gamma=fusion_grid.best_params_["svc__gamma"],
                cache_size=512,
                random_state=SEED,
                verbose=1,
            ),
        ),
    ]
)
fusion_svm.fit(combined[train_idx], y_all[train_idx])
fusion_val = float(
    accuracy_score(y_all[val_idx], fusion_svm.predict(combined[val_idx]))
)
print("Validation tujuh fitur:", fusion_val)
print("Selisih terhadap tiga embedding:", fusion_val - baseline_val)

[LibSVM]*
optimization finished, #iter = 475
obj = -26.502536, rho = 0.034043
nSV = 285, nBSV = 0
*
optimization finished, #iter = 551
obj = -34.601131, rho = 0.079406
nSV = 333, nBSV = 3
*
optimization finished, #iter = 516
obj = -29.146000, rho = 0.122902
nSV = 308, nBSV = 0
*
optimization finished, #iter = 460
obj = -24.196137, rho = 0.004831
nSV = 286, nBSV = 0
*
optimization finished, #iter = 462
obj = -23.581144, rho = 0.008512
nSV = 286, nBSV = 0
*
optimization finished, #iter = 464
obj = -23.938676, rho = -0.000383
nSV = 297, nBSV = 0
*
optimization finished, #iter = 493
obj = -25.381796, rho = 0.065680
nSV = 307, nBSV = 0
*
optimization finished, #iter = 503
obj = -28.151409, rho = -0.048312
nSV = 296, nBSV = 0
*
optimization finished, #iter = 488
obj = -28.234796, rho = 0.064813
nSV = 305, nBSV = 0
*
optimization finished, #iter = 510
obj = -28.080129, rho = 0.055880
nSV = 313, nBSV = 0
*
optimization finished, #iter = 523
obj = -33.086955, rho = 0.114222
nSV = 325, nBSV = 2


## 9. Simpan kedua model
Metadata hash dan split mencegah Testing memasangkan SVM dengan CNN lain.

In [9]:
joblib.dump({"pipeline": baseline_svm, "run_id": RUN_ID, "split_hash": split_hash,
             "model_hashes": model_hashes, "feature_groups": 3},
            OUT / "model" / "baseline_svm.joblib")
joblib.dump({"pipeline": fusion_svm, "run_id": RUN_ID, "split_hash": split_hash,
             "model_hashes": model_hashes, "feature_groups": 7,
             "boundaries": boundaries.tolist(), "preprocessing_version": "v1"},
            OUT / "model" / "fusion_svm.joblib")

['/home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v1/fusion/model/fusion_svm.joblib']

## 10. Simpan laporan pencarian
Catat seluruh kandidat, waktu nyata, hasil pembanding, dan perubahan akurasi validation.

In [10]:
def grid_summary(grid, seconds):
    return {"best_params": grid.best_params_, "best_validation_accuracy": float(grid.best_score_),
            "seconds": seconds, "candidates": [{"params": p, "score": float(s)} for p, s in
            zip(grid.cv_results_["params"], grid.cv_results_["mean_test_score"])]}
report = {"run_id": RUN_ID, "split_hash": split_hash,
          "three_embeddings": grid_summary(baseline_grid, baseline_seconds),
          "seven_groups": grid_summary(fusion_grid, fusion_seconds),
          "baseline_final_validation_accuracy": baseline_val,
          "fusion_final_validation_accuracy": fusion_val,
          "validation_delta": fusion_val - baseline_val}
(OUT / "laporan" / "fusion_grid.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(
    "Target official test belum dievaluasi; "
    "jalankan Testing setelah semua model final tersimpan."
)

Target official test belum dievaluasi; jalankan Testing setelah semua model final tersimpan.
